# Notebook 06 - Leakage-Safe Feature Engineering

Stage 7 creates dated pre-GARCH features from the validated aligned SPY/VIX data. It does not estimate GARCH, identify regimes, construct transition outcomes, or fit statistical models.

SPY **Adjusted Close** is used consistently for close-to-close returns and drawdowns because it reflects price changes after corporate actions. Raw Close is retained as a source field but is not mixed into these calculations.

## Definitions

For adjusted price $P_t$, simple return is $P_t/P_{t-1}-1$ and log return is $\log(P_t/P_{t-1})$. Close-to-open is an intraday measure, so it is not the primary daily return.

The volume benchmark is $\operatorname{mean}(Volume_{t-20},\ldots,Volume_{t-1})$: today is shifted out before the 20-day rolling mean. Abnormal volume is current volume divided by that prior-only benchmark. A value of 1.00 equals the prior average; 1.50 is 50% above it; and 0.70 is 30% below it.

Drawdown is current adjusted price divided by the highest price observed through today in a full 252-day or 60-day window, minus one. It is therefore zero at a rolling high and otherwise negative.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'configs' / 'research_config.yaml').exists():
    raise FileNotFoundError('Run this notebook from the notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from market_regime.config import load_research_config
from market_regime.features import build_pre_garch_features, save_feature_checkpoint


In [2]:
config = load_research_config(PROJECT_ROOT / 'configs' / 'research_config.yaml')
aligned_path = PROJECT_ROOT / 'data' / 'processed' / 'spy_vix_aligned_raw.csv'
output_path = PROJECT_ROOT / 'data' / 'processed' / 'market_features_pre_garch.csv'
raw_aligned = pd.read_csv(aligned_path, index_col='Date', parse_dates=True)
raw_aligned.index.name = 'Date'

if 'Adj_Close' not in raw_aligned.columns:
    raise ValueError('Adj_Close is unavailable. Stage 7 does not silently substitute Close.')
print('Price column used consistently for returns and drawdown: Adj_Close')
print('Validated Stage 6 input rows:', len(raw_aligned))


Price column used consistently for returns and drawdown: Adj_Close
Validated Stage 6 input rows: 6641


In [3]:
features = build_pre_garch_features(
    raw_aligned,
    volume_window=config['features']['abnormal_volume_window'],
    primary_drawdown_window=config['features']['primary_drawdown_window'],
    robustness_drawdown_window=config['features']['robustness_drawdown_window'],
    training_end=config['data']['training_end'],
    test_start=config['data']['test_start'],
)
save_feature_checkpoint(features, output_path)
print('Saved:', output_path)
print('Output rows:', len(features))
display(features.head())
display(features.tail())


Saved:

 C:\Users\kalyan\Downloads\sp-500-clustering\sp-500-regime-clustering\data\processed\market_features_pre_garch.csv
Output rows: 6641


,Adj_Close,Close,High,Low,Open,Volume,VIX_Adj_Close,VIX_Close,VIX_High,VIX_Low,...,Log_Return,Volume_MA20_Previous,Abnormal_Volume,Log_Abnormal_Volume,Rolling_Peak_252,Drawdown_252,Rolling_Peak_60,Drawdown_60,Sample,Is_Train
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,91.132751,145.4375,148.25000,143.875000,148.25000,8164300,24.209999,24.209999,26.150000,23.980000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Train,True
2000-01-04,87.568909,139.7500,144.06250,139.640625,143.53125,8089800,27.010000,27.010000,27.180000,24.799999,...,-0.039891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Train,True
2000-01-05,87.725540,140.0000,141.53125,137.250000,139.93750,12177900,26.410000,26.410000,29.000000,25.850000,...,0.001787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Train,True
2000-01-06,86.315666,137.7500,141.50000,137.750000,139.62500,6227200,25.730000,25.730000,26.709999,24.700001,...,-0.016202,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Train,True
2000-01-07,91.328560,145.7500,145.75000,140.062500,140.31250,8066500,21.719999,21.719999,25.170000,21.719999,...,0.056452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Train,True


,Adj_Close,Close,High,Low,Open,Volume,VIX_Adj_Close,VIX_Close,VIX_High,VIX_Low,...,Log_Return,Volume_MA20_Previous,Abnormal_Volume,Log_Abnormal_Volume,Rolling_Peak_252,Drawdown_252,Rolling_Peak_60,Drawdown_60,Sample,Is_Train
Date,,,,,,,,,,,,,,,,,,,,,
2026-05-22,743.723999,745.640015,748.940002,744.479980,746.239990,41762000,16.700001,16.700001,17.389999,16.459999,...,0.003924,47701875.0,0.875479,-0.132984,746.247498,-0.003382,746.247498,-0.003382,Test,False
2026-05-26,748.661316,750.590027,752.130005,748.369995,750.010010,41123600,17.010000,17.010000,17.230000,16.559999,...,0.006617,47530875.0,0.865198,-0.144797,748.661316,0.000000,748.661316,0.000000,Test,False
2026-05-27,748.531616,750.460022,751.380005,748.219971,750.880005,42106300,16.290001,16.290001,17.180000,16.290001,...,-0.000173,47930260.0,0.878491,-0.129550,748.661316,-0.000173,748.661316,-0.000173,Test,False
2026-05-28,752.660950,754.599976,755.150024,749.229980,750.250000,41562600,15.740000,15.740000,16.850000,15.610000,...,0.005501,47879705.0,0.868063,-0.141491,752.660950,0.000000,752.660950,0.000000,Test,False
2026-05-29,754.536133,756.479980,758.080017,754.690002,755.900024,55075700,15.320000,15.320000,15.880000,15.220000,...,0.002488,47864875.0,1.150650,0.140327,754.536133,0.000000,754.536133,0.000000,Test,False


In [4]:
engineered_columns = ['Simple_Return', 'Log_Return', 'Volume_MA20_Previous', 'Abnormal_Volume', 'Log_Abnormal_Volume', 'Rolling_Peak_252', 'Drawdown_252', 'Rolling_Peak_60', 'Drawdown_60']
print('Missing-value counts (initial full-window values are intentionally missing):')
display(features[engineered_columns].isna().sum().to_frame('missing_count'))

summary_columns = ['Simple_Return', 'Log_Return', 'Abnormal_Volume', 'Log_Abnormal_Volume', 'Drawdown_252', 'Drawdown_60', 'VIX_Close']
print('Summary statistics:')
display(features[summary_columns].describe().T)
print('Train/Test counts:')
display(features['Sample'].value_counts().to_frame('rows'))


Missing-value counts (initial full-window values are intentionally missing):


,missing_count
Simple_Return,1
Log_Return,1
Volume_MA20_Previous,20
Abnormal_Volume,20
Log_Abnormal_Volume,20
Rolling_Peak_252,251
Drawdown_252,251
Rolling_Peak_60,59
Drawdown_60,59


Summary statistics:


,count,mean,std,min,25%,50%,75%,max
Simple_Return,6640.0,0.000392,0.012167,-0.109424,-0.004578,0.000694,0.005995,0.145197
Log_Return,6640.0,0.000318,0.012171,-0.115887,-0.004589,0.000694,0.005977,0.135577
Abnormal_Volume,6621.0,1.018205,0.374134,0.171545,0.775047,0.947620,1.171247,5.149965
Log_Abnormal_Volume,6621.0,-0.039113,0.332431,-1.762911,-0.254832,-0.053802,0.158069,1.638990
Drawdown_252,6390.0,-0.063435,0.089463,-0.514815,-0.087915,-0.022965,-0.004199,0.000000
Drawdown_60,6582.0,-0.036199,0.049765,-0.417108,-0.050993,-0.016424,-0.002785,0.000000
VIX_Close,6641.0,19.836071,8.316316,9.140000,14.030000,17.820000,23.209999,82.690002


Train/Test counts:


,rows
Sample,
Train,4528
Test,2113


In [5]:
demonstration = features[['Volume', 'Volume_MA20_Previous', 'Abnormal_Volume']].dropna().head(3)
print('The benchmark uses only previous dates; today\'s volume is not included:')
display(demonstration)
assert features['Abnormal_Volume'].dropna().gt(0).all()
assert features[['Drawdown_252', 'Drawdown_60']].max().max() <= 1e-12
assert (features['Log_Return'].dropna() - __import__('numpy').log1p(features['Simple_Return'].dropna())).abs().max() < 1e-12
print('Positivity, drawdown, and return-identity checks passed.')


The benchmark uses only previous dates; today's volume is not included:


,Volume,Volume_MA20_Previous,Abnormal_Volume
Date,,,
2000-02-01,8419900,7838540.0,1.074167
2000-02-02,6205900,7851320.0,0.790428
2000-02-03,7997500,7757125.0,1.030988


Positivity, drawdown, and return-identity checks passed.


## Stage 7 boundary

The saved file is a complete dated pre-GARCH feature checkpoint. Missing initial feature values are preserved rather than filled or dropped. GARCH conditional volatility, standardization, K-means, HMM states, transitions, ANOVA, and regression have not been estimated.